In [7]:
# ============================================================
"""
Script: pinns_L2_vs_k.py

Description:
    Train PINNs for different wavenumbers k and compute
    relative L2 error vs k.
"""
# ============================================================

#%% ======================== IMPORTS ========================
from datetime import datetime
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Paths
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utilities')
os.chdir(current_dir)
sys.path.insert(0, utilities_dir)

# Custom imports
from analytical_solution_functions import sound_hard_circle_calc, mask_displacement
from pinns_solution_functions import (
    set_seed, generate_points, MLP, init_weights,
    train_adam_with_logs, train_lbfgs_with_logs,
    predict_displacement_pinns
)

set_seed(42)

In [8]:
#%% ======================== PARAMETERS ========================

r_i = np.pi / 4
l_e = np.pi
side_length = 2 * l_e
n_grid = 501

n_Omega_P = 10_000
n_Gamma_I = 100
n_Gamma_E = 250

# Training
adam_lr = 1e-2
adam_iters = 1000
lbfgs_iters = 3000   # reduce for speed

hidden_layers_ = 3
hidden_units_  = 25

# Study range
k_values = np.linspace(1, 10, 10)
l2_errors = []

# Grid
Y, X = np.mgrid[-l_e:l_e:n_grid*1j, -l_e:l_e:n_grid*1j]
R_exact = np.sqrt(X**2 + Y**2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Activation
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)

activation_function_ = Sine()

In [9]:
 


#%% ======================== LOOP OVER k ========================

for k in k_values:

    print(f"\nTraining PINN for k = {k:.2f}")

    # -------- Analytical solution --------
    #_, u_scn_exact, _ = sound_hard_circle_calc(k, r_i, X, Y)
    #u_scn_exact = mask_displacement(R_exact, r_i, l_e, u_scn_exact)

 
    u_inc_exact, u_scn_exact, u_exact = sound_hard_circle_calc(k, r_i, X, Y, n_terms=None)
    u_inc_exact = mask_displacement(R_exact, r_i, l_e, u_inc_exact)
    u_scn_exact = mask_displacement(R_exact, r_i, l_e, u_scn_exact)
    u_exact = mask_displacement(R_exact, r_i, l_e, u_exact)


    # -------- Training points --------
    x_f, y_f, x_inner, y_inner, x_left, y_left, x_right, y_right, \
    x_bottom, y_bottom, x_top, y_top = generate_points(
        n_Omega_P, side_length, r_i, n_Gamma_I, n_Gamma_E
    )
 
    # -------- Model --------
    model = MLP(
        input_size=2,
        output_size=2,
        hidden_layers=hidden_layers_,
        hidden_units=hidden_units_,
        activation_function=activation_function_
    ).to(device)

    model.apply(init_weights)

    results = []
    iter_train = 0

    # -------- Adam --------
    iter_train = train_adam_with_logs(
        model,
        x_f, y_f,
        x_inner, y_inner,
        x_left, y_left,
        x_right, y_right,
        x_bottom, y_bottom,
        x_top, y_top,
        k,
        iter_train,
        results,
        adam_lr,
        num_iter=adam_iters,
        save_csv_path=None,
        save_csv_path_no_datetime=None,
        l_e=l_e,
        r_i=r_i,
        n_grid=n_grid,
        X=X,
        Y=Y,
        R_exact=R_exact,
        u_scn_exact=u_scn_exact,
        u_exact=u_exact
    )

    # -------- L-BFGS --------
    train_lbfgs_with_logs(
        model,
        x_f, y_f,
        x_inner, y_inner,
        x_left, y_left,
        x_right, y_right,
        x_bottom, y_bottom,
        x_top, y_top,
        k,
        iter_start=iter_train,
        results=results,
        lbfgs_lr=1.0,
        num_iter=lbfgs_iters,
        save_csv_path=None,
        save_csv_path_no_datetime=None,
        l_e=l_e,
        r_i=r_i,
        n_grid=n_grid,
        X=X,
        Y=Y,
        R_exact=R_exact,
        u_scn_exact=u_scn_exact,
        u_exact=u_exact
    )

    # -------- Prediction --------
    u_sc_amp_pinns, _, _, _ = predict_displacement_pinns(
        model, l_e, r_i, k, n_grid
    )

    # -------- L2 error --------
    u_exact_masked = np.copy(u_scn_exact)
    u_pred_masked  = np.copy(u_sc_amp_pinns)

    u_exact_masked[R_exact < r_i] = 0
    u_pred_masked[R_exact < r_i] = 0

    rel_L2 = np.linalg.norm(u_exact_masked.real - u_pred_masked.real, 2) / \
             np.linalg.norm(u_exact_masked.real, 2)

    print(f"k = {k:.2f}, PINNs L2 error = {rel_L2:.3e}")

    l2_errors.append(rel_L2)

np.save("data/pinns_L2_errors_wavenumbers.npy", np.array(l2_errors))

#%% ======================== PLOT ========================
 
os.makedirs("data", exist_ok=True)
date_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

np.save(f"data/pinns_L2_errors_wavenumbers_{date_str}.npy", np.array(l2_errors))

print("\nSaved L2 errors to:")
print(f"data/pinns_L2_errors_wavenumbers_{date_str}.npy")
 


Training PINN for k = 1.00
Adam - Iter: 1 - Loss: 4.557898998260498 - Mean Rel Error: 1.3939409238409506
Adam - Iter: 100 - Loss: 0.1106957197189331 - Mean Rel Error: 1.8715118259759924
Adam - Iter: 200 - Loss: 0.09383605420589447 - Mean Rel Error: 1.8960813762963198
Adam - Iter: 300 - Loss: 0.08883386105298996 - Mean Rel Error: 1.8280670977880735
Adam - Iter: 400 - Loss: 0.08140645921230316 - Mean Rel Error: 1.6371425056786344
Adam - Iter: 500 - Loss: 0.06601422280073166 - Mean Rel Error: 1.2354493922095484
Adam - Iter: 600 - Loss: 0.04841732978820801 - Mean Rel Error: 0.5460582603909511
Adam - Iter: 700 - Loss: 0.014443891122937202 - Mean Rel Error: 0.22231248067767437
Adam - Iter: 800 - Loss: 0.009191271848976612 - Mean Rel Error: 0.15809008691063564
Adam - Iter: 900 - Loss: 0.006476684007793665 - Mean Rel Error: 0.13302040611116858
Adam - Iter: 1000 - Loss: 0.005284147337079048 - Mean Rel Error: 0.14538205039089297
LBFGS - Iter: 1100 - Loss: 0.0018370819743722677 - Mean Rel Error: